In [2]:
import logging
import pandas as pd
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.llms import HuggingFacePipeline
from langchain.vectorstores import FAISS
from transformers import pipeline

# 로깅 설정
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def extract_helpful_answer(result):
    marker = "Helpful Answer :"
    start_index = result['result'].find(marker)
    if start_index != -1:
        return result['result'][start_index + len(marker):].strip()
    else:
        return "Helpful Answer: 구분자가 문자열에 없습니다."

def generate_talent_description(company_name, retriever, llm):
    template = """
        너는 지금부터 회사의 기업보고서를 보고 인재상을 출력해줄거야
        아래의 방식대로 기업의 인재상을 출력해줘

        1. 주어진 정보를 바탕으로 기업의 인재상을 최대한 모두 작성해줘.
        2. 인재상 설명 이외의 불필요한 내용은 포함하지 마. 오직 인재상만 출력해줘.
        3. 주어진 정보만으로 인재상을 추론하기 어려운 경우 '인재상을 추론하기에 정보가 부족합니다.'라고 말해줘.
        4. 모든 답변은 한국어로 작성해줘.

        문서 : {context}

        Question : {question}

        (출력예시)
        1. 인재상1 ...
        2. 인재상2 ...
        3. 인재상3 ...
        ================
        Helpful Answer :  
    """
    
    QA_CHAIN_PROMPT = PromptTemplate.from_template(template)
    question = f"{company_name}의 인재상을 작성해줘."
    
    qa_chain = RetrievalQA.from_chain_type(
        llm,
        retriever=retriever,
        return_source_documents=True,
        chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
    )
    
    result = qa_chain({'query': question})
    talents = extract_helpful_answer(result)
    return talents

def read_csv(csv_file_path):
    try:
        return pd.read_csv(csv_file_path)
    except FileNotFoundError:
        logger.warning(f"{csv_file_path} not found. Creating a new file.")
        return pd.DataFrame(columns=["company_name", "talents"])

def save_to_csv(df, csv_file_path):
    df.to_csv(csv_file_path, index=False)
    logger.info(f"Data saved to {csv_file_path}")

def create_vector_store(company_info, embeddings):
    return FAISS.from_texts(texts=[company_info], embedding=embeddings)

def main():
    company_name = "삼성SDS"
    csv_file_path = "./company_talents.csv"
    
    df = read_csv(csv_file_path)
    
    hf_pipeline = pipeline("text-generation", model=model_c4ai, tokenizer=tokenizer_c4ai, streamer=streamer, max_new_tokens=100, do_sample=True, temperature=0.1)
    llm = HuggingFacePipeline(pipeline=hf_pipeline)
    
    if company_name in df["company_name"].values:
        talents = df[df["company_name"] == company_name]["talents"].tolist()
        logger.info(f"Talents: {talents}")
    else:
        company_info = extract_company_info(company_name)
        if company_info:
            vector_store = create_vector_store(company_info, embeddings)
            retriever = vector_store.as_retriever(search_kwargs={"k": 1})
            
            talents = generate


<module 'time' (built-in)>